In [2]:
import pandas as pd

df_ground_truth = pd.read_csv('../../../data/ground_truth-new.csv')
ground_truth = df_ground_truth.to_dict(orient='records')

In [3]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = [x for x in documents if x['course'] == 'llm-zoomcamp']

documents = documents_llm
index = build_index(documents)

In [5]:
def text_search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}

    return index.search(
        query, 
        num_results=5, 
        boost_dict=boost_dict
    )

In [6]:
q = ground_truth[0]
q

{'question': 'Can I still join the course if I just found it now?',
 'document': '74eb249bbf'}

In [7]:
doc_id = q['document']
results = text_search(query=q['question'])

In [8]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
a9353fadfe == 74eb249bbf: False
9f689c185f == 74eb249bbf: False
977bf7786c == 74eb249bbf: False
69d122f12e == 74eb249bbf: False


In [9]:
relevance = []

for d in results:
    relevance.append(int(d['id'] == doc_id))

relevance

[1, 0, 0, 0, 0]

In [10]:
def compute_relevance_text(q):
    doc_id = q['document']
    results = text_search(query=q['question'])

    relevance = []

    for d in results:
        relevance.append(int(d['id'] == doc_id))

    return relevance

In [11]:
q = ground_truth[0]
print(q['question'])
compute_relevance_text(q)

Can I still join the course if I just found it now?


[1, 0, 0, 0, 0]

In [12]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

/var/home/alexis/src/llm-zoomcamp/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
ground_truth_sample = ground_truth[:15]
relevant_total_text = compute_relevance_total_text(ground_truth_sample)

100%|██████████| 15/15 [00:00<00:00, 359.80it/s]


In [14]:
relevant_total_text

[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

In [15]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [16]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [17]:
relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

100%|██████████| 15/15 [00:00<00:00, 369.25it/s]


[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

In [18]:
relevance_total = compute_relevance_total(ground_truth, text_search)

100%|██████████| 565/565 [00:01<00:00, 424.23it/s]


In [19]:
relevance_total

[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 1, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [1, 0, 0, 0, 0],
 [1, 0, 0,

#### Search Evaluation Metrics

In [21]:
# Hit Rate
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [22]:
# Mean Reciprocal Rank (MRR)
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [23]:
# Putting them together
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [24]:
evaluate(
    ground_truth,
    text_search
)

100%|██████████| 565/565 [00:01<00:00, 422.86it/s]


{'hit_rate': 0.8495575221238938, 'mrr': 0.7313274336283183}

#### Search Parameter Tuning

In [25]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [26]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

100%|██████████| 565/565 [00:01<00:00, 417.93it/s]


boost=0.5: {'hit_rate': 0.879646017699115, 'mrr': 0.7856342182890853}


100%|██████████| 565/565 [00:01<00:00, 424.50it/s]


boost=1.0: {'hit_rate': 0.8955752212389381, 'mrr': 0.795870206489675}


100%|██████████| 565/565 [00:01<00:00, 426.34it/s]


boost=3.0: {'hit_rate': 0.8495575221238938, 'mrr': 0.7313274336283183}


100%|██████████| 565/565 [00:01<00:00, 432.54it/s]


boost=5.0: {'hit_rate': 0.8212389380530973, 'mrr': 0.6983775811209437}


100%|██████████| 565/565 [00:01<00:00, 430.68it/s]

boost=10.0: {'hit_rate': 0.7911504424778761, 'mrr': 0.666371681415929}


In [27]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [28]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={question_boost},"
                f" answer_boost={answer_boost},"
                f" section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


100%|██████████| 565/565 [00:01<00:00, 412.35it/s]


Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


100%|██████████| 565/565 [00:01<00:00, 424.97it/s]


Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

100%|██████████| 565/565 [00:01<00:00, 431.49it/s]


Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


100%|██████████| 565/565 [00:01<00:00, 431.93it/s]


Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


100%|██████████| 565/565 [00:01<00:00, 428.44it/s]


In [29]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
7,1.0,4.0,0.2,0.976991,0.872802
6,1.0,4.0,0.1,0.976991,0.872773
18,2.0,4.0,0.1,0.969912,0.870619
8,1.0,4.0,0.5,0.971681,0.869646
34,5.0,10.0,0.2,0.968142,0.869381
19,2.0,4.0,0.2,0.968142,0.869381
35,5.0,10.0,0.5,0.968142,0.869381
3,1.0,2.0,0.1,0.968142,0.869381
4,1.0,2.0,0.2,0.962832,0.868732
20,2.0,4.0,0.5,0.961062,0.868643


In [30]:
def text_search(query):
    boost_dict = {
        "question": 1.0,
        "answer": 4.0,
        "section": 0.3,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )